## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE WITH ADDITIONAL COVARIATES 
This script creates the phenotype file for SAIGE-GENE+. It integrates several inputs, including EUR-like exome sample quality control results, ancestry-specific PCs, and age and sex files. Only individuals who pass exome quality control are retained in the SAIGE-GENE+ phenotype file.
Case–control status is assigned by providing a list of person IDs for individuals designated as cases; all remaining individuals are assigned as controls. The script also restricts the dataset to individuals who have electronic health record data. All quantitative covariates are standardized to have a mean of 0 and a standard deviation of 1.
Because more samples passed exome quality control than ACAF quality control in this analysis, this phenotype file will also be used for the ACAF analysis, with samples failing ACAF quality control excluded at a later stage.

In [ ]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

# Set environment variable for workspace bucket 
my_bucket = 'gs://rw-migration-aou-rw-9f879fc4'
%run /home/jupyter/workspace/rw-migration-aou-rw-9f879fc4/notebooks/Setting_Env_Variables.ipynb

In [ ]:
# Load quality control information and demographic information 
bucket=f'{my_bucket}/data/pcs/EUR_flashpca.pcs'
eur_pca = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/covariates/biosample_collection_age.tsv'
age = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt'
eur_exome = pd.read_csv(bucket, sep ='\t')

bucket=f'{my_bucket}/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt'
sex = pd.read_csv(bucket, sep ='\t')

bucket=f'gs://rw-migration-aou-rw-9f879fc4/data/phenotype/aml_person_ids.txt'
aml_person_df = pd.read_csv(bucket, sep ='\t')

bucket=f'gs://rw-migration-aou-rw-9f879fc4/data/phenotype/ehr_person_ids.txt'
ehr_person_df = pd.read_csv(bucket, sep ='\t')

# Assign biospecimen variable 
sex['biospecimen'] = sex['sample_source'].apply(lambda x: 1 if x == "WHOLE BLOOD" else 0 if x == "SALIVA" else 2)
sex['biospecimen'] = sex['biospecimen'].astype("int")

# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

# Create list of quality control information and demographic information 
dfs = [eur_exome.rename(columns={"IID":"person_id"}),
       age.rename(columns={"IID":"person_id"}).filter(regex="person_id|age"),
       sex.rename(columns={"IID":"person_id"}).filter(regex="person_id|sex|biospecimen"), 
       eur_pca.rename(columns={"IID":"person_id"}).filter(regex="person_id|PC"),
       ehr_person_df
       ]

# Combine list of quality control information and demographic information 
phenotype_file = reduce(
    lambda left, right: pd.merge(left, right, on="person_id", how="inner"),
    dfs
)

# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

# Additional covariates 
phenotype_file["age_squared"] = phenotype_file["age"] ** 2
phenotype_file["age_sex"] = phenotype_file["age"] * phenotype_file["sex"]
phenotype_file["age_biospecimen"] = phenotype_file["age"] * phenotype_file["biospecimen"]


# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype',
            'sex', 'age', 'biospecimen',
            'age_squared', 'age_sex', 'age_biospecimen', 
            'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
            'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
            'person_id']]
)

# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col 
                       in phenotype_file.columns 
                       if col.startswith("PC") 
                       or col.startswith("age")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = (
    scaler.fit_transform(phenotype_file[cols_to_standardize])
) 
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_phenotype"].value_counts())
print("============================================")

# Save phentype file 
phenotype_file.to_csv(
    'EUR_phenotype_file.txt',
    sep='\t',
    index=False
)

In [ ]:
# Copy phenotype file to workspace bucket 
!gcloud storage cp EUR_phenotype_file.txt gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/aml_covariates_files/